<a href="https://colab.research.google.com/github/ReesavGupta/AI-First-Internal-Helpdesk-Portal/blob/master/Reesav_FineTuning_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y transformers accelerate datasets peft
!pip install -U torch torchvision torchaudio
!pip install -U transformers[torch] accelerate datasets peft

Found existing installation: transformers 4.55.4
Uninstalling transformers-4.55.4:
  Successfully uninstalled transformers-4.55.4
Found existing installation: accelerate 1.10.1
Uninstalling accelerate-1.10.1:
  Successfully uninstalled accelerate-1.10.1
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: peft 0.17.1
Uninstalling peft-0.17.1:
  Successfully uninstalled peft-0.17.1
  Using cached accelerate-1.10.1-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.17.1-py3-none-any.whl.metadata (14 kB)
  Using cached transformers-4.55.4-py3-none-any.whl.metadata (41 kB)
Using cached accelerate-1.10.1-py3-none-any.whl (374 kB)
Using cached datasets-4.0.0-py3-none-any.whl (494 kB)
Using cached peft-0.17.1-py3-none-any.whl (504 kB)
Using cached transformers-4.55.4-py3-none-any.whl (11.3 MB)


In [2]:
!pip install -U transformers peft accelerate bitsandbytes datasets evaluate scikit-learn sentencepiece wandb bitsandbytes

In [2]:
!pip install transformers

!pip install peft

!pip install accelerate

In [3]:
!pip show transformers peft accelerate | grep Version

Version: 4.55.4
Version: 0.17.1
Version: 1.10.1


In [10]:
import os, json
import torch
import numpy as np
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding, set_seed)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

# -----------------------
# Config
# -----------------------
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
METHOD = "lora"  # Keep it simple
OUTPUT_DIR = f"runs/{METHOD}-tinyllama"
SEED = 42

# Simple training params
LR = 2e-4
BATCH = 8
EPOCHS = 3

# LoRA params
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

set_seed(SEED)

# -----------------------
# Load Data
# -----------------------
print("Loading Banking77 dataset...")
raw = load_dataset("banking77")
raw = raw.rename_column("text", "sentence")

# Split train into train/val
split = raw["train"].train_test_split(test_size=0.1, seed=SEED)
train_ds, val_ds, test_ds = split["train"], split["test"], raw["test"]

num_labels = len(raw["train"].features["label"].names)
print(f"Dataset loaded: {len(train_ds)} train, {len(val_ds)} val, {len(test_ds)} test")
print(f"Number of labels: {num_labels}")

# -----------------------
# Tokenizer & Preprocessing
# -----------------------
print(f"Loading tokenizer for {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=256, padding=False)

train_tok = train_ds.map(preprocess, batched=True, remove_columns=["sentence"])
val_tok = val_ds.map(preprocess, batched=True, remove_columns=["sentence"])
test_tok = test_ds.map(preprocess, batched=True, remove_columns=["sentence"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -----------------------
# Model
# -----------------------
print(f"Loading model: {MODEL_NAME}")
config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels = num_labels
config.problem_type = "single_label_classification"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    torch_dtype=torch.float32   # <- force full precision
)

# Fix pad token
if tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id

# -----------------------
# Apply LoRA
# -----------------------
print(f"Applying LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}")
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules="all-linear",
    bias="none"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# -----------------------
# Metrics
# -----------------------
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = evaluate.load("accuracy")
    f1 = evaluate.load("f1")

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    }

# -----------------------
# Training - Minimal Args
# -----------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Very minimal TrainingArguments to avoid compatibility issues
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    num_train_epochs=EPOCHS,
    logging_steps=50,
    save_steps=500,
    eval_steps=500,
    fp16=False,  # Disable FP16 to avoid gradient scaling issues
    dataloader_drop_last=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n" + "="*50)
print(f"Starting training with LoRA method")
print("="*50)

# Train
trainer.train()

# -----------------------
# Test Evaluation
# -----------------------
print("\nEvaluating on test set...")
test_results = trainer.evaluate(eval_dataset=test_tok)

print("\n" + "="*50)
print("FINAL TEST RESULTS:")
for key, value in test_results.items():
    if key.startswith("eval_"):
        metric_name = key.replace("eval_", "").upper()
        print(f"{metric_name}: {value:.4f}")
print("="*50)

# Save results
results_file = os.path.join(OUTPUT_DIR, "test_results.json")
with open(results_file, "w") as f:
    json.dump(test_results, f, indent=2)

# Save adapter
adapter_dir = os.path.join(OUTPUT_DIR, "adapter")
model.save_pretrained(adapter_dir)
print(f"\nAdapter saved to: {adapter_dir}")

print("\nTraining completed successfully! 🎉")

Loading Banking77 dataset...
Dataset loaded: 9002 train, 1001 val, 3080 test
Number of labels: 77
Loading tokenizer for TinyLlama/TinyLlama-1.1B-Chat-v1.0


Map:   0%|          | 0/9002 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Applying LoRA: rank=16, alpha=32
trainable params: 12,773,376 || all params: 1,047,443,456 || trainable%: 1.2195


/tmp/ipython-input-1797824132.py:129: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



Starting training with LoRA method
{'loss': 4.8511, 'grad_norm': 21.776891708374023, 'learning_rate': 0.0001970988750740083, 'epoch': 0.04440497335701599}
{'loss': 3.177, 'grad_norm': 18.870588302612305, 'learning_rate': 0.0001941385435168739, 'epoch': 0.08880994671403197}
{'loss': 1.5396, 'grad_norm': 20.67327308654785, 'learning_rate': 0.00019117821195973952, 'epoch': 0.13321492007104796}
{'loss': 1.1686, 'grad_norm': 11.22295093536377, 'learning_rate': 0.0001882178804026051, 'epoch': 0.17761989342806395}
{'loss': 0.9612, 'grad_norm': 17.352703094482422, 'learning_rate': 0.0001852575488454707, 'epoch': 0.22202486678507993}
{'loss': 0.7535, 'grad_norm': 12.663628578186035, 'learning_rate': 0.0001822972172883363, 'epoch': 0.2664298401420959}
{'loss': 0.8118, 'grad_norm': 11.869938850402832, 'learning_rate': 0.00017933688573120192, 'epoch': 0.3108348134991119}
{'loss': 0.6935, 'grad_norm': 7.540285110473633, 'learning_rate': 0.0001763765541740675, 'epoch': 0.3552397868561279}
{'loss': 

In [11]:
!zip -r adapter_lora.zip runs/lora-tinyllama/adapter

# Download to your local computer
from google.colab import files
files.download("adapter_lora.zip")

  adding: runs/lora-tinyllama/adapter/ (stored 0%)
  adding: runs/lora-tinyllama/adapter/README.md (deflated 66%)
  adding: runs/lora-tinyllama/adapter/adapter_model.safetensors (deflated 8%)
  adding: runs/lora-tinyllama/adapter/adapter_config.json (deflated 57%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>